## Lock in the real-world graph
In this notebook, we set up a basic simulation where two vessels cross a lock complex in a real-world graph. We take the eastern lock chamber of the Volkerak sluices (rightern most lock chamber in the picture below).

![OpenTNSim-Volkeraksluizen](figures/0206_Volkeraksluizen.png)


#### 0. Import
##### 0.1 Import libraries

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package to navigate through files and folders
import os

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
import opentnsim.fis as fis
from opentnsim.graph.calculations import (transform_geometry, calculate_bounding_rectangle, flip_coordinates, split_edge_based_on_geometry_along_edge,
                                          calculate_length_of_splitted_edge_geometries, calculate_the_distances_from_doors_to_edge_nodes, 
                                          calculate_object_dimensions_and_alignment, split_edge_based_on_point_along_edge)
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.graph.utils import (align_network_geometries_with_edge_directions, find_edges_in_a_polygon, get_closest_node_to_point, 
                                   get_closest_edge_to_point, get_closest_location_on_edge_to_point)
from opentnsim.graph.visualizations import create_real_world_graph
from opentnsim.output import HasOutput

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable

# package(s) needed for inspecting the output
import pandas as pd
import matplotlib.pyplot as plt

# package(s) needed for inspecting the output
import pandas as pd
import geopandas as gpd
import numpy as np

# package needed to download the lock geometry
import requests

# plot libraries
import folium

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


In [2]:
%load_ext autoreload
%autoreload 2

##### 0.2. Functions to be included in the utilities

#### 1. Define object classes

In [3]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
    {}
)

#### 2 Create graph
##### 2.1 Import FIS graph
Next we create a network (a graph) along which the vessel can move. For this case we use the Fairway Information System graph, and make the vessels sail from one side of the lock to another side.

In [4]:
# load the processed version from the Fairway Information System graph provided by Rijkswaterstaat
FG = fis.load_network(version="0.3")

From the above map we select the following origin and destination pair for the vessels

In [5]:
node_A = '8860743'
node_B = '8866727'

##### 2.2 Modify the FIS graph
We have to check if there are nodes within the lock chamber geometry. This is not supported by OpenTNSim, and we need to manually remove these nodes and merge the edges to a single edge. Let's first import the lock chamber geometry, which we downloaded using OpenStreetMap (OSM) through overpass turbo (see [link](https://overpass-turbo.eu/s/2chr)).

In [6]:
# Use faster server
url = "https://overpass.kumi.systems/api/interpreter"

# Berlin bounding box (faster than area lookup)
query = """
[out:json][timeout:180];
nwr["lock_name"~"Oostkolk"](51.68005,4.38958,51.70062,4.42933);
out geom;
"""

response = requests.get(url, params={'data': query})
data = response.json()

# Convert to DataFrame
elements = data["elements"]

df = pd.json_normalize(elements)

# Create geometry column
df['geometry'] = df.apply(lambda row: Polygon((p["lon"], p["lat"]) for p in row.geometry), axis=1)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

gdf.head()

,type,id,nodes,geometry,bounds.minlat,bounds.minlon,bounds.maxlat,bounds.maxlon,tags.CEMT,tags.boat,...,tags.maxwidth,tags.name,tags.natural,tags.phone,tags.url,tags.vhf,tags.water,tags.wikidata,tags.wikipedia,tags.waterway
0,way,56198966,"[4515246328, 8536258682, 8536258681, 451524632...","POLYGON ((4.40799 51.68904, 4.40801 51.68903, ...",51.688860,4.407993,51.691111,4.412082,VIb,yes,...,24.1,Volkeraksluizen,water,+31 88 797 4990,https://youtu.be/VHbaJfyRVzU,7;25;64,lock,Q1886412,nl:Volkeraksluizen,NaN
1,way,251906187,"[4515246327, 2580607082, 4297997740, 451524632...","POLYGON ((4.40807 51.68892, 4.40823 51.68901, ...",51.688925,4.408070,51.691053,4.411981,VIb,yes,...,24.1,Volkeraksluizen,NaN,+31 88 797 4990,NaN,7;25;64,NaN,Q1886412,nl:Volkeraksluizen,canal


In [7]:
lock_geometry = gdf.geometry.iloc[0]

We can plot this geometry into the folium map by reconfiguring the coordinates of the lock chamber's polygon geometry:

In [8]:
sub_FG = FG.subgraph(nx.dijkstra_path(FG, node_A, node_B))
m = create_real_world_graph(sub_FG, lat_start = 51.69, lon_start = 4.41, zoom_start = 14)
folium_lock_coords = [[lat, lon] for lon, lat in lock_geometry.exterior.coords]
folium.Polygon(folium_lock_coords).add_to(m)

#display
m

There are nodes within the lock geometry, which we have to find, merge their edges and remove.

In [9]:
from opentnsim.graph.utils import find_nodes_in_a_polygon
from opentnsim.graph.calculations import merge_two_consecutive_edges_based_on_shared_node

nodes_within_the_lock_chamber = find_nodes_in_a_polygon(FG, lock_geometry)
for node in nodes_within_the_lock_chamber:
    merge_two_consecutive_edges_based_on_shared_node(FG,node)

In [10]:
sub_FG = FG.subgraph(nx.dijkstra_path(FG, node_A, node_B))
m = create_real_world_graph(sub_FG, lat_start = 51.69, lon_start = 4.41, zoom_start = 14)
folium.Polygon(folium_lock_coords).add_to(m)

#display
m

In [11]:
total_length = 0
for i,edge in enumerate(sub_FG.edges):
    edge_geometry = sub_FG.edges[edge]["geometry"]
    edge_geometry_m = transform_geometry(edge_geometry, epsg_out = "EPSG:28992")
    edge_geometry_length = edge_geometry_m.length
    sub_FG.edges[edge]["length_m"] = edge_geometry_length
    total_length += edge_geometry_length

sub_FG = align_network_geometries_with_edge_directions(sub_FG)

In [12]:
sub_FG_test = sub_FG.to_directed()

In [13]:
sub_FG_test = align_network_geometries_with_edge_directions(sub_FG_test)

In [14]:
import opentnsim.graph.mixins as graph_module
graph_module.plot_graph(sub_FG_test)

##### 2.3 Derive lock information
First identify at which edge the lock chamber is located:

In [15]:
lock_node_A, lock_node_B = find_edges_in_a_polygon(sub_FG,lock_geometry)[0]
lock_edge = (lock_node_A, lock_node_B)

Next, we need to derive the lock dimensions, which we can do using a bounding box polygon

In [16]:
bounding_rectange_transformed = calculate_bounding_rectangle(lock_geometry)
bounding_rectange = flip_coordinates(bounding_rectange_transformed) 
bounding_rectange_m = transform_geometry(bounding_rectange, epsg_out = "EPSG:28992")

Distances from the gates to the edge's start and end node

In [17]:
# split the lock edge in three parts
first_part_edge, lock_part_edge, second_part_edge = split_edge_based_on_geometry_along_edge(FG,lock_edge,bounding_rectange)

# determine the length of the different parts
edge_geometries = [first_part_edge,lock_part_edge,second_part_edge]
edge_lenghts = calculate_length_of_splitted_edge_geometries(FG, lock_edge, edge_geometries)

# select the geometries and length of the part of the edges that do not contain the lock chamber itself
lock_edge_geometries = [first_part_edge, second_part_edge]
lock_edge_lengths = [edge_lenghts[0],edge_lenghts[-1]]

# determine the distances of the edge parts
(distance_from_start_node_to_lock_doors_A, 
 distance_from_end_node_to_lock_doors_B) = calculate_the_distances_from_doors_to_edge_nodes(FG, lock_edge, lock_edge_geometries, lock_edge_lengths)

The lock dimensions

In [18]:
# we already calculated the length of the lock chamber
lock_length = edge_lenghts[1]

# we wrote a function to calculate the other dimensions of the lock chamber
_, lock_width, rotation = calculate_object_dimensions_and_alignment(bounding_rectange_m)

# as the length also include two mitre lock gates (length of two times half the lock width) and a safety marging, lets say 15 m at both lock doors, the length capacity would be:
lock_length_capacity = np.floor((lock_length - lock_width - 2*15)/10)*10 # ~301.5 m in reality
lock_width_capacity = np.floor(lock_width) # 24.1 m in reality
lock_depth = 6.2  

The waiting areas and their distances to the lock doors

In [19]:
# roughly determine the waiting area coordinates using google earth
waiting_area_A_geometry = Point(4.420425,51.696497)
waiting_area_B_geometry = Point(4.395560,51.683439)

# get the closest nodes and edges where the waiting areas are located
node_waiting_area_A = get_closest_node_to_point(FG,waiting_area_A_geometry)
node_waiting_area_B = get_closest_node_to_point(FG,waiting_area_B_geometry)
edge_waiting_area_A = get_closest_edge_to_point(FG,waiting_area_A_geometry)
edge_waiting_area_B = get_closest_edge_to_point(FG,waiting_area_B_geometry)

# get routes to the lock
route_to_lock_edge_from_waiting_area_A = nx.dijkstra_path(FG,node_waiting_area_A,lock_node_A)
route_to_lock_edge_from_waiting_area_B = nx.dijkstra_path(FG,node_waiting_area_B,lock_node_B)

# get the location along this edge at which the waiting area is located
waiting_area_location_on_edge_waiting_area_A = get_closest_location_on_edge_to_point(FG, edge_waiting_area_A, waiting_area_A_geometry)
waiting_area_location_on_edge_waiting_area_B = get_closest_location_on_edge_to_point(FG, edge_waiting_area_B, waiting_area_B_geometry)

# get the splitted edges
edge_waiting_area_A_geometry_splitted = split_edge_based_on_point_along_edge(FG,edge_waiting_area_A,waiting_area_location_on_edge_waiting_area_A)
edge_waiting_area_B_geometry_splitted = split_edge_based_on_point_along_edge(FG,edge_waiting_area_B,waiting_area_location_on_edge_waiting_area_B)

# determine their lengths
edge_lenghts_waiting_area_A = calculate_length_of_splitted_edge_geometries(FG, edge_waiting_area_A, edge_waiting_area_A_geometry_splitted)
edge_lenghts_waiting_area_B = calculate_length_of_splitted_edge_geometries(FG, edge_waiting_area_B, edge_waiting_area_B_geometry_splitted)

C:\Users\floorbakker\Anaconda3\envs\opentnsim\lib\site-packages\shapely\measurement.py:72: RuntimeWarning:

invalid value encountered in distance



In [20]:
# determine distance to node directed to the lock chamber
edge_waiting_area_B_in_route_to_lock = edge_waiting_area_B[0] in route_to_lock_edge_from_waiting_area_B and edge_waiting_area_B[1] in route_to_lock_edge_from_waiting_area_B
distance_lock_doors_B_to_waiting_area_B = 0.0
if edge_waiting_area_B_in_route_to_lock:
    distance_lock_doors_B_to_waiting_area_B += np.max(edge_lenghts_waiting_area_B)
    remaining_route_to_lock_doors_B = route_to_lock_edge_from_waiting_area_B[1:]
else:
    distance_lock_doors_B_to_waiting_area_B += np.min(edge_lenghts_waiting_area_B)
    remaining_route_to_lock_doors_B = route_to_lock_edge_from_waiting_area_B
if len(remaining_route_to_lock_doors_B) >= 2:
    for edge_start_node, edge_end_node in zip(remaining_route_to_lock_doors_B[:-1],remaining_route_to_lock_doors_B[1:]):
        edge_info = FG.edges[(edge_start_node,edge_end_node)]
        distance_lock_doors_B_to_waiting_area_B += edge_info["length_m"]
if edge_waiting_area_B != lock_edge:
    distance_lock_doors_B_to_waiting_area_B += distance_from_end_node_to_lock_doors_B
else:
    distance_lock_doors_B_to_waiting_area_B = np.max(edge_lenghts_waiting_area_B) - distance_from_end_node_to_lock_doors_A - lock_length

In [21]:
edge_waiting_area_A_in_route_to_lock = edge_waiting_area_A[0] in route_to_lock_edge_from_waiting_area_A and edge_waiting_area_A[1] in route_to_lock_edge_from_waiting_area_A
distance_lock_doors_A_to_waiting_area_A = 0.0
if edge_waiting_area_A_in_route_to_lock:
    distance_lock_doors_A_to_waiting_area_A += np.max(edge_lenghts_waiting_area_A)
    remaining_route_to_lock_doors_A = route_to_lock_edge_from_waiting_area_A[1:]
else:
    distance_lock_doors_A_to_waiting_area_A += np.min(edge_lenghts_waiting_area_A)
    remaining_route_to_lock_doors_A = route_to_lock_edge_from_waiting_area_A
if len(remaining_route_to_lock_doors_A) >= 2:
    for edge_start_node, edge_end_node in zip(remaining_route_to_lock_doors_A[:-1],remaining_route_to_lock_doors_A[1:]):
        edge_info = FG.edges[(edge_start_node,edge_end_node)]
        distance_lock_doors_A_to_waiting_area_A += edge_info["length_m"]
if edge_waiting_area_A != lock_edge:
    distance_lock_doors_A_to_waiting_area_A += distance_from_start_node_to_lock_doors_A
else:
    distance_lock_doors_A_to_waiting_area_A = np.max(edge_lenghts_waiting_area_A) - distance_from_end_node_to_lock_doors_B - lock_length

#### 3. Run simulation
##### 3.1 Set mission and simpy environment with VTS

In [22]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [23]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

# add graph to environment
env.graph = sub_FG_test

##### 3.2 Create lock object

In [24]:
lock_chamber = IsLockChamber(env=env,
                             name='Oostkolk',
                             geometry = lock_geometry,
                             lock_depth = lock_depth,
                             gate_open = lock_node_A,
                             edge = (lock_node_A,lock_node_B),
                             crs_m='EPSG:28992') #Amersfoort / RD New reference system

359.48421576


In [25]:
lock_chamber.lock_length,lock_chamber.length.level

(359.48421576, 359.48421576)

In [26]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = edge_waiting_area_A,
                                   distance_from_edge_start = np.min(edge_lenghts_waiting_area_A))

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = edge_waiting_area_B,
                                   distance_from_edge_start = np.min(edge_lenghts_waiting_area_B))

In [27]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = [node_A,node_B],
                             env=env,
                             name = 'Lock complex',)

In [28]:
sub_FG = FG.subgraph(nx.dijkstra_path(FG, node_A, node_B))
m = create_real_world_graph(sub_FG, lat_start = 51.69, lon_start = 4.41, zoom_start = 14)
folium_lock_coords = [[lat, lon] for lon, lat in lock_chamber.geometry.exterior.coords]
folium_waA_coords = [[lat, lon] for lon, lat in waiting_area_A.geometry.coords][0]
folium_waB_coords = [[lat, lon] for lon, lat in waiting_area_B.geometry.coords][0]
folium.Circle(folium_waA_coords,radius=10).add_to(m)
folium.Circle(folium_waB_coords,radius=10).add_to(m)
folium.Polygon(folium_lock_coords).add_to(m)

#display
m

##### 3.3 Create vessels

In [29]:
# create vessels from dict 
data_vessel_1 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 1",                                  # required by Identifiable
    "geometry": env.graph.nodes[node_A]['geometry'],     # required by Locatable
    "route": nx.dijkstra_path(env.graph, node_A, node_B),# required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 5,                                              # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
}  
vessel_1 = Vessel(**data_vessel_1)
vessel_1.name = 'Vessel 1'

data_vessel_2 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 2",                                  # required by Identifiable
    "geometry": env.graph.nodes[node_B]['geometry'],     # required by Locatable
    "route": nx.dijkstra_path(env.graph, node_B, node_A),# required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 5,                                              # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:05:00')  # required by PassesLockComplex
}  
vessel_2 = Vessel(**data_vessel_2)
vessel_2.name = 'Vessel 2'

# start the simulation
env.process(mission(env, vessel_1))
env.process(mission(env, vessel_2))
env.run()

In [30]:
# We can plot the time-distance diagram
lock_chamber.plot(xlimmin = -3000, 
                  xlimmax = 3000,
                  method = 'Plotly')

In [31]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel_1.logbook)

print("'{}' logbook data:".format(vessel_1.name))  
print('')

display(df)

'Vessel 1' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 8860743 to node 8862498 start,2025-01-01 00:00:00.000000,0,POINT (4.43615944214078 51.7022764335403)
1,Sailing from node 8860743 to node 8862498 stop,2025-01-01 00:01:59.152925,476.611698,POINT (4.43027504290525 51.7000442261129)
2,Sailing from node 8862498 to node B34113_A start,2025-01-01 00:01:59.152925,476.611698,POINT (4.43027504290525 51.7000442261129)
3,Sailing to first lock gate start,2025-01-01 00:01:59.152925,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.43027504290525 51.7000442261129)
4,Sailing to first lock gate stop,2025-01-01 00:05:16.849165,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.411997212001874 51.691033759393214)
5,Sailing to position in lock start,2025-01-01 00:05:16.849165,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.411997212001874 51.691033759393214)
6,Sailing to position in lock stop,2025-01-01 00:10:17.643759,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.408608545334134 51.68921656579189)
7,Levelling start,2025-01-01 00:22:10.450423,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.408608545334134 51.68921656579189)
8,Levelling stop,2025-01-01 00:32:10.450423,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.408608545334134 51.68921656579189)
9,Sailing to second lock gate start,2025-01-01 00:37:10.450423,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.408608545334134 51.68921656579189)


In [32]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel_2.logbook)

print("'{}' logbook data:".format(vessel_2.name))  
print('')

display(df)

'Vessel 2' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 8866727 to node 8868426 start,2025-01-01 00:05:00.000000,0,POINT (4.38801816186906 51.678798010768)
1,Sailing from node 8866727 to node 8868426 stop,2025-01-01 00:06:43.756196,415.024785,POINT (4.39253941691783 51.6812485487213)
2,Sailing from node 8868426 to node B34113_B start,2025-01-01 00:06:43.756196,415.024785,POINT (4.39253941691783 51.6812485487213)
3,Waiting for lock operation start,2025-01-01 00:06:43.756196,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.39253941691783 51.6812485487213)
4,Waiting for lock operation stop,2025-01-01 00:32:59.541662,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.39253941691783 51.6812485487213)
5,Sailing from node 8868426 to node B34113_B stop,2025-01-01 00:36:06.909605,1164.496555,POINT (4.407620736969263 51.688686277276325)
6,Sailing from node B34113_B to node B34113_A start,2025-01-01 00:36:06.909605,1164.496555,POINT (4.407620736969263 51.688686277276325)
7,Sailing from node B34113_B to node B34113_A stop,2025-01-01 00:36:08.922243,1172.547108,POINT (4.4077088440752465 51.688733575668216)
8,Sailing from node B34113_A to node 8862498 start,2025-01-01 00:36:08.922243,1172.547108,POINT (4.4077088440752465 51.688733575668216)
9,Sailing to first lock gate start,2025-01-01 00:36:08.922243,"{'origin': '', 'destination': '', 'route': [],...",POINT (4.4077088440752465 51.688733575668216)
